# Chest X-Ray Experiment Suite on Colab

This is the preferred Colab workflow for the refactored project.

It will:
- install dependencies
- clone or open the project
- download the Kaggle dataset
- validate it with `prepare.py`
- run a sequence of experiment configs
- aggregate the resulting metrics
- package outputs for download


## 1. Runtime

Use `Runtime -> Change runtime type -> T4 GPU` in Colab before running.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/aya')
REPO_URL = ''  # Optional: set this to your GitHub repo URL.
REPO_BRANCH = 'main'
CONFIGS_TO_RUN = [
    'configs/model_cnn.yaml',
    'configs/model_effnet_frozen.yaml',
    'configs/model_effnet_finetune.yaml',
    'configs/model_lstm32.yaml'
]
DATASET_DIR = PROJECT_DIR / 'data' / 'raw' / 'chest_xray'

print('PROJECT_DIR =', PROJECT_DIR)
print('CONFIGS_TO_RUN =', CONFIGS_TO_RUN)
print('DATASET_DIR =', DATASET_DIR)


In [ ]:
!nvidia-smi || true
!python3 --version
!pip -q install -U pip
!pip -q install kaggle


In [ ]:
if PROJECT_DIR.exists():
    print(f'Using existing project at {PROJECT_DIR}')
elif REPO_URL:
    !git clone --branch "$REPO_BRANCH" "$REPO_URL" "$PROJECT_DIR"
else:
    raise RuntimeError(
        'Set REPO_URL in the config cell, or upload the repo to /content/aya before continuing.'
    )

%cd $PROJECT_DIR
!pip -q install -r requirements.txt
!find . -maxdepth 2 -type f | sort | sed -n '1,140p'


In [ ]:
from google.colab import files
import os
from pathlib import Path

print('Upload kaggle.json from your Kaggle account.')
uploaded = files.upload()
if 'kaggle.json' not in uploaded:
    raise RuntimeError('kaggle.json was not uploaded.')

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_path = kaggle_dir / 'kaggle.json'
kaggle_path.write_bytes(uploaded['kaggle.json'])
os.chmod(kaggle_path, 0o600)

download_root = PROJECT_DIR / 'data' / 'raw'
download_root.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p "$download_root"
!unzip -qo "$download_root/chest-xray-pneumonia.zip" -d "$download_root"

nested_dir = download_root / 'chest_xray' / 'chest_xray'
flat_dir = download_root / 'chest_xray'

if nested_dir.exists():
    source_dir = nested_dir
elif flat_dir.exists():
    source_dir = flat_dir
else:
    raise RuntimeError('Could not find extracted chest_xray dataset directory.')

if DATASET_DIR.exists():
    print(f'Dataset already exists at {DATASET_DIR}')
else:
    DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
    if source_dir != DATASET_DIR:
        os.rename(source_dir, DATASET_DIR)

print('Dataset ready at', DATASET_DIR)
!find "$DATASET_DIR" -maxdepth 2 -type d | sort


In [ ]:
%cd $PROJECT_DIR
!python3 prepare.py

config_args = ' '.join(CONFIGS_TO_RUN)
!PYTHONPATH=src python3 experiments/run_suite.py $config_args


In [ ]:
import json
from pathlib import Path
import pandas as pd
import yaml

def resolve_config(config_path: Path):
    raw = yaml.safe_load(config_path.read_text()) or {}
    if 'base_config' not in raw:
        return raw
    base = yaml.safe_load((config_path.parent / raw['base_config']).read_text()) or {}
    merged = dict(base)
    for key, value in raw.items():
        if key == 'base_config':
            continue
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = {**merged[key], **value}
        else:
            merged[key] = value
    return merged

rows = []
for config_name in CONFIGS_TO_RUN:
    cfg = resolve_config(PROJECT_DIR / config_name)
    experiment_name = cfg.get('name', Path(config_name).stem)
    metrics_path = PROJECT_DIR / 'outputs' / 'runs' / experiment_name / 'metrics.json'
    metrics = json.loads(metrics_path.read_text())
    rows.append({
        'experiment': experiment_name,
        'val_macro_f1': metrics['validation']['macro_f1'],
        'val_accuracy': metrics['validation']['accuracy'],
        'val_loss': metrics['validation']['loss'],
        'test_macro_f1': metrics['test']['macro_f1'],
        'test_accuracy': metrics['test']['accuracy'],
        'test_loss': metrics['test']['loss'],
        'train_time_seconds': metrics['train_time_seconds']
    })

results_df = pd.DataFrame(rows).sort_values(['val_macro_f1', 'val_accuracy'], ascending=False)
results_df


In [ ]:
summary_path = PROJECT_DIR / 'outputs' / 'colab_suite_summary.csv'
results_df.to_csv(summary_path, index=False)
print('Saved summary to', summary_path)
summary_path


In [ ]:
from google.colab import files
import shutil

archive_base = str(PROJECT_DIR / 'outputs' / 'colab_suite_artifacts')
archive_file = shutil.make_archive(archive_base, 'zip', root_dir=PROJECT_DIR / 'outputs')
print('Created', archive_file)
files.download(archive_file)
